# Papyrus-Layoytstudien

In [1]:
from pathlib import Path
import sys
import xml.etree.ElementTree as ET
import json

# Das Notebook liegt im Unterordner python/. Verzeichnis robust bestimmen,
# egal ob der Kernel aus python/ oder aus der Projektwurzel gestartet wird.
SCRIPT_DIR = Path.cwd()
if SCRIPT_DIR.name != 'python' and (SCRIPT_DIR / 'python').is_dir():
    SCRIPT_DIR = SCRIPT_DIR / 'python'
sys.path.insert(0, str(SCRIPT_DIR))   # Module (page_xml_to_json etc.) importierbar machen

PROJECT_ROOT = SCRIPT_DIR.parent       # Projektwurzel, eine Ebene ueber python/
PAGE_XML_DIR = PROJECT_ROOT / 'page_xml'
DATA_FILE = PROJECT_ROOT / 'layout_data.json'

ns = {'p': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

# re_page_xml = re.compile(r"^P.*\.xml$")

# PAGE XML to JSON

Die Auswertung der PAGE-XML-Dateien ist in das Modul `page_xml_to_json.py` ausgelagert. Es erfasst pro Platte Bildmaße, px/cm-Skala und die Textregionen – getrennt nach Kolumnen (`column_data`) und Fragmenten (`fragment_data`) – und schreibt das Ergebnis nach `DATA_FILE`.

Die Liste der auszuwertenden Dateien (`page_xml_files`) wird oben im Notebook definiert und als Argument an `collect_layout_data(...)` übergeben. Bestehende Einträge in der JSON bleiben erhalten (z. B. manuelle bbox-Felder und Messwerte); neue Platten kommen automatisch hinzu.

In [2]:
# find page xml in `page` subfolders of `page_xml_dir`
page_xml_files = list(PAGE_XML_DIR.rglob('page/*.xml'))
for file in page_xml_files:
    print(file)

c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0001_P_09782-Pl-A_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0002_P_09782-Pl-B_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0003_P_09782-Pl-C_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0004_P_09782-Pl-D_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0005_P_09782-Pl-E_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0006_P_09782-Pl-F_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0007_P_09782-Pl-G1_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0008_P_09782-Pl-H_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0009_P_09782-Pl-N2_R_001.xml
c:\Users\Fabi\Repos\papyrus-layout-studien\page_xml\P_09782\P_09782\page\0010_P_09782-Pl-

In [3]:
import page_xml_to_json as pxj

# `page_xml_files` wird oben definiert und hier als Argument uebergeben.
layout_data = pxj.collect_layout_data(page_xml_files, data_file=DATA_FILE)

Processing page: 0001_P_09782-Pl-A_R_001
Processing page: 0002_P_09782-Pl-B_R_001
Processing page: 0003_P_09782-Pl-C_R_001
Processing page: 0004_P_09782-Pl-D_R_001
Processing page: 0005_P_09782-Pl-E_R_001
Processing page: 0006_P_09782-Pl-F_R_001
Processing page: 0007_P_09782-Pl-G1_R_001
Processing page: 0008_P_09782-Pl-H_R_001
Processing page: 0009_P_09782-Pl-N2_R_001
Processing page: 0010_P_09782-Pl-O_R_001
Processing page: 0011_P_09782-Pl-P_R_001
Processing page: 0012_P_09782-Pl-Q_R_001
Processing page: 0013_P_09782-Pl-R_R_001
Processing page: 0014_P_09782-Pl-S_R_001
Processing page: 0015_P_09782-Pl-T_R_001
Writing 15 pages (62 columns, 4 fragments) to c:\Users\Fabi\Repos\papyrus-layout-studien\layout_data.json


# Image Processing

Die Bildverarbeitung ist in das Modul `image_processing.py` ausgelagert. Es segmentiert das Fragment vom hellen Hintergrund, erzeugt eine Binärmaske und leitet daraus Bounding-Box, Flächen- und Randmaße ab. Der Aufruf erfolgt weiter unten über `process_layout_data(...)`.

## Einstellbare Parameter

Alle Stellschrauben sind in der Dataclass `SegmentationParams` gebündelt und können beim Aufruf überschrieben werden, z. B. `ip.SegmentationParams(min_component_ratio=0.05)`.

- **`strip_threshold_factor`** (Default `0.85`): Erkennung des Info-Streifens am unteren Rand. Wird der Streifen nicht sauber entfernt, Wert senken.
- **`gray_close_kernel_size`** (Default `(51, 51)`): Überbrückt helle Stellen im Fragment. Wird das Fragment von hellen Flecken „aufgefressen“, vergrößern.
- **`binary_fill_kernel_size`** (Default `(15, 15)`): Füllt kleine Löcher und Risse in der Maske.
- **`min_component_ratio`** (Default `0.02`): Rauschfilter. Bei vielen kleinen Artefakten (wie in `P_09782-Pl-H_R_001`) erhöhen (z. B. `0.05` oder `0.1`), um nur die größten Teile zu behalten.

Mit `overwrite_masks=True` werden zwischengespeicherte Masken neu berechnet; ansonsten werden vorhandene Masken aus `images/masks/` wiederverwendet.

## Manual Bounding Box Mode

For fragments where automatic segmentation is unreliable, you can manually define the bounding box in `layout_data.json`. The processor will then use this data instead of running the image segmentation pipeline.

### How to Use

Add these fields to the relevant entry in `layout_data.json`:

```json
{
  "image_key": {
    "bbox_measurement": "manual",
    "bbox_manual": {
      "origin": [x, y],
      "width": w,
      "height": h
    },
    ...rest of data...
  }
}
```

**Fields:**
- `bbox_measurement`: Set to `"manual"` to enable manual mode; set to `"automatic"` or omit for automatic segmentation
- `bbox_manual.origin`: `[x, y]` - top-left corner of the bounding box
- `bbox_manual.width`: `w` - width in pixels
- `bbox_manual.height`: `h` - height in pixels

### Behavior

When `bbox_measurement` is set to `"manual"`:
- The image segmentation pipeline is **skipped**
- No binary mask is created or saved
- All derived measurements (margins, coverage) are computed from the manual bbox
- `coverage_pct` is set to 100% (assuming the entire bbox is fragment)
- The preview image is still generated with the bbox overlaid

### Example

```json
"0008_P_09782-Pl-H_R_001": {
  "bbox_measurement": "manual",
  "bbox_manual": {
    "origin": [120, 80],
    "width": 1200,
    "height": 1500
  },
  ...
}
```

In [ ]:
import image_processing as ip

# Segmentierungsparameter (bei Bedarf anpassen, siehe Dokumentation oben).
params = ip.SegmentationParams()

# Pipeline ausfuehren: verarbeitet alle Eintraege, schreibt die Masse zurueck
# nach DATA_FILE und legt Masken/Vorschaubilder unter images/masks/ ab.
layout_data = ip.process_layout_data(
    layout_data,
    project_root=PROJECT_ROOT,
    data_file=DATA_FILE,
    params=params,
    overwrite_masks=False,  # True regeneriert alle Masken
)

# Kolumnen-Merkmale

Einfache Layout-Merkmale je Kolumne und Kolumnenpaar, ausgelagert in `column_metrics.py`. Läuft **nach** der Bounding-Box-Berechnung, da oberer/unterer Rand und die Schriftspiegel-Verhältnisse das Blattmaß (Bbox) brauchen.

Pro Kolumne: Höhe, Breite, Fläche, oberer/unterer Rand, Schriftspiegel-Verhältnis (Höhe & Fläche). Pro Nachbarpaar (in `to_next` der linken Kolumne): Intercolumn-Abstand, Kolumne-zu-Kolumne-Breite und -Fläche. Werte in px und – wo eine Skala vorliegt – cm/cm²; Ränder und Schriftspiegel werden bei Mehrfach-Fragment-Platten (z. B. T) unterdrückt.

Horizontale Abstände werden in der Mitte des y-Überlappungsbereichs benachbarter Kanten gemessen (= Mittel über die Überlappung, da die Kanten gerade sind).

In [ ]:
import column_metrics as cm

# Einfache Merkmale je Kolumne/Paar berechnen (nach der Bbox-Berechnung) und
# nach DATA_FILE zurueckschreiben.
layout_data = cm.measure_columns(layout_data, data_file=DATA_FILE)

# Kolumnenneigung (Maas's Law)

Neigungswinkel der linken Kante jeder **nutzbaren** Kolumne, ausgelagert in `column_tilt.py`. Zwei Varianten (vgl. Methodik-Notiz „Winkelmessung Maas's Law"):

1. `tilt_vs_vertical_deg` – gegen die reine Bild-Senkrechte (Annahme: das Digitalisat ist lotrecht ausgerichtet).
2. `tilt_vs_ideal_deg` – gegen die Senkrechte einer plattenweiten Ideal-Horizontalen, gemittelt aus den **Oberkanten** der nutzbaren Kolumnen (entfernt eine globale Plattenschiefe).

Ergebnis je Kolumne in `column['tilt']`, je Platte in `entry['tilt_reference']` (u. a. `plate_skew_deg` = geschätzte Plattenschiefe = Differenz beider Varianten). Vorzeichen: **positiv = untere Kante nach links versetzt**. Erwartung: ~1–4° pro Kolumne.

In [10]:
import column_tilt as ct

# Kolumnenneigung (Maas's Law) je nutzbarer Kolumne berechnen und nach
# DATA_FILE zurueckschreiben.
layout_data = ct.measure_tilt(layout_data, data_file=DATA_FILE)

[0001_P_09782-Pl-A_R_001] 5 nutzbare Kolumnen, Plattenschiefe 0.842°
[0002_P_09782-Pl-B_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.909°
[0003_P_09782-Pl-C_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.286°
[0004_P_09782-Pl-D_R_001] 4 nutzbare Kolumnen, Plattenschiefe -0.045°
[0005_P_09782-Pl-E_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.203°
[0006_P_09782-Pl-F_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.238°
[0007_P_09782-Pl-G1_R_001] 2 nutzbare Kolumnen, Plattenschiefe 0.603°
[0008_P_09782-Pl-H_R_001] 11 nutzbare Kolumnen, Plattenschiefe -0.165°
[0009_P_09782-Pl-N2_R_001] 2 nutzbare Kolumnen, Plattenschiefe -0.464°
[0010_P_09782-Pl-O_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.442°
[0011_P_09782-Pl-P_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.39°
[0012_P_09782-Pl-Q_R_001] 4 nutzbare Kolumnen, Plattenschiefe 0.79°
[0013_P_09782-Pl-R_R_001] 2 nutzbare Kolumnen, Plattenschiefe -0.727°
[0014_P_09782-Pl-S_R_001] 1 nutzbare Kolumnen, Plattenschiefe -0.357°

55 Kolumnen vermessen -> c:

# Datenüberblick (Markdown-Export)

Erzeugt aus `layout_data.json` einen lesbaren Überblick und speichert ihn als `layout_overview.md`. Die Logik ist in das Modul `layout_overview.py` ausgelagert; `generate_overview(...)` lädt die Daten, baut die Tabelle und schreibt die Datei.

Pro Platte: Anzahl Kolumnen (gesamt und nutzbar), Fragmente, Höhe/Breite (cm) sowie Zeilen-Kennzahlen. Durchschnitt, Minimum und Maximum beziehen sich nur auf Kolumnen mit erfassten Zeilen; Fragmente werden separat gezählt; Höhe/Breite (cm) entfallen ohne Skala oder bei mehreren Fragmenten.

In [ ]:
import layout_overview as lo
from IPython.display import Markdown, display

# Baut den Markdown-Ueberblick aus DATA_FILE und schreibt layout_overview.md.
markdown = lo.generate_overview(
    data_file=DATA_FILE,
    overview_file=PROJECT_ROOT / 'layout_overview.md',
)
display(Markdown(markdown))